
DNA26-1410


In [0]:
%sql
create or replace table `p-cdh`.reference.biteam_bp_net_recoveries_key as select * except(BusinessDate, LoadingDateTime) from `p-cdh`.landed.biteam_bp_net_recoveries_key_vw where businessdate = '2026-10-30';

create or replace table `p-cdh`.reference.biteam_bp_gross_recoveries_key as select * except(BusinessDate, LoadingDateTime) from `p-cdh`.landed.biteam_bp_gross_recoveries_key_vw where businessdate = '2026-10-30';

create or replace table `p-cdh`.reference.biteam_hermes_investor_bp TBLPROPERTIES ('delta.columnMapping.mode' = 'name') as select * except(BusinessDate, LoadingDateTime) from `p-cdh`.landed.biteam_hermes_investor_bp_vw where businessdate = '2026-10-30';

create or replace table `p-cdh`.reference.biteam_neptune_investor_bp TBLPROPERTIES ('delta.columnMapping.mode' = 'name') as select * except(BusinessDate, LoadingDateTime) from `p-cdh`.landed.biteam_neptune_investor_bp_vw where businessdate = '2026-10-30';

create or replace table `p-cdh`.reference.biteam_whl_glx_cs_haps_targets_per_borrower TBLPROPERTIES ('delta.columnMapping.mode' = 'name') as select * except(BusinessDate, LoadingDateTime) from `p-cdh`.landed.biteam_whl_glx_cs_haps_targets_per_borrower_vw where businessdate = '2026-10-30';

create or replace table `p-cdh`.reference.biteam_bp_recoveries_strategy as select * except(BusinessDate, LoadingDateTime) from `p-cdh`.landed.biteam_bp_recoveries_strategy where businessdate = '2026-10-30';

create or replace table `p-cdh`.reference.biteam_bp_amicable_report as select * except(BusinessDate, LoadingDateTime) from `p-cdh`.landed.biteam_bp_amicable_report where businessdate = '2026-10-30';

DNA26-1391_Dag

In [0]:
import os, boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("newsphone_arc_backfill").getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

# ---- AWS creds from secret scope ----
SECRETS_SCOPE = "aws-databricks"
_keys = dbutils.secrets.list(scope=SECRETS_SCOPE)
_aws_key = [k.key for k in _keys][0]
_aws_secret = dbutils.secrets.get(scope=SECRETS_SCOPE, key=_aws_key)
os.environ["AWS_ACCESS_KEY_ID"] = _aws_key
os.environ["AWS_SECRET_ACCESS_KEY"] = _aws_secret
os.environ["AWS_DEFAULT_REGION"] = "eu-central-1"

# ---- config (only ENV_PREFIX changes per env) ----
ENV_PREFIX   = "p"
CATALOG      = f"`{ENV_PREFIX}-cdh`"
SCHEMA_LNG   = "`landing`"
SCHEMA_LND   = "`landed`"
TABLE_NAME   = "newsphone_auction_report_content"
FOLDER_NAME  = "newsphone_auction_report_content"
SOURCE_BUCKET    = f"{ENV_PREFIX}-ec1-cepal-landing-zone"
INCOMING_PREFIX  = "incoming/newsphone/"
PROCESSED_PREFIX = "incoming/newsphone/processed/"
FILE_AREA    = "AuctionReportContent"

cdh_location_lng = f"{CATALOG}.{SCHEMA_LNG}.{FOLDER_NAME}"
cdh_location_lnd = f"{CATALOG}.{SCHEMA_LND}.{TABLE_NAME}"

s3 = boto3.client("s3", region_name="eu-central-1")

def list_incoming_arc_files():
    pg = s3.get_paginator("list_objects_v2")
    keys = []
    for page in pg.paginate(Bucket=SOURCE_BUCKET, Prefix=INCOMING_PREFIX, Delimiter="/"):
        for o in page.get("Contents", []):
            f = o["Key"].split("/")[-1]
            if f.startswith(FILE_AREA) and f.endswith(".csv"):
                keys.append(f)
    return keys

print("setup done | env:", ENV_PREFIX)


In [0]:
if spark.catalog.tableExists(cdh_location_lng):
    print(f"Landing table already exists: {cdh_location_lng}")
else:
    spark.sql(f"""
        CREATE TABLE {cdh_location_lng} (
            UniqueCode STRING,
            FileContent STRING,
            CorruptRecordFlag BOOLEAN,
            file_date DATE,
            file_name STRING,
            Cluster_0 STRING
        )
        USING delta
        CLUSTER BY (Cluster_0)
        TBLPROPERTIES (
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.columnMapping.mode' = 'name'
        )
    """)
    print(f"Landing table created: {cdh_location_lng}")


In [0]:
# Read ALL AuctionReportContent CSVs in one distributed read
src_glob = f"s3://{SOURCE_BUCKET}/{INCOMING_PREFIX}AuctionReportContent_*.csv"

raw = (
    spark.read
    .option("header", "true")
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")   # both cols are strings; skip inference
    .csv(src_glob)
    .withColumn("_src", F.expr("_metadata.file_path"))
)

# derive file_date (yyyy-MM-dd) and Cluster_0 (yyyyMMdd) from the filename
raw = (
    raw
    .withColumn("_fname", F.regexp_extract("_src", r"(AuctionReportContent_\d{8}\.csv)$", 1))
    .withColumn("_ymd",   F.regexp_extract("_fname", r"_(\d{8})\.csv$", 1))
    .withColumn("file_date", F.to_date("_ymd", "yyyyMMdd"))
    .withColumn("file_name", F.col("_fname"))
    .withColumn("Cluster_0", F.col("_ymd"))                 # bare yyyyMMdd, matches loader
    .withColumn("CorruptRecordFlag", F.lit(False))
)

# strip stray quotes like the loader's regexp_replace(col, '"', "")
for c in ["UniqueCode", "FileContent"]:
    raw = raw.withColumn(c, F.regexp_replace(F.col(c), '"', ""))

landing_df = raw.select(
    "UniqueCode", "FileContent", "CorruptRecordFlag", "file_date", "file_name", "Cluster_0"
)

(landing_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cdh_location_lng))

print("LANDING DONE (bulk)")
spark.sql(f"SELECT COUNT(DISTINCT file_date) dates, MIN(file_date), MAX(file_date), COUNT(*) rows FROM {cdh_location_lng}").show(truncate=False)


In [0]:
lng = spark.table(cdh_location_lng)

# latest business date across all files
latest_date = lng.agg(F.max("file_date")).collect()[0][0]
print("latest_date:", latest_date)

# one row per UniqueCode (earliest occurrence), stamped with the latest BusinessDate
w = Window.partitionBy("UniqueCode").orderBy(F.col("file_date").asc())
latest_only = (
    lng.withColumn("_rn", F.row_number().over(w))
       .filter("_rn = 1")
       .withColumn("BusinessDate", F.lit(latest_date).cast("date"))
       .withColumn("LoadingDateTime", F.from_utc_timestamp(F.current_timestamp(), "Europe/Athens"))
       .select("UniqueCode", "FileContent", "CorruptRecordFlag",
               "file_date", "file_name", "BusinessDate", "LoadingDateTime")
)

(latest_only.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("BusinessDate")
    .saveAsTable(cdh_location_lnd))


print("LANDED committed")

In [0]:
# Move all AuctionReportContent_* files from incoming/ to incoming/processed/
to_move = list_incoming_arc_files()
print(f"Moving {len(to_move)} files to processed/")

for i, fname in enumerate(to_move, start=1):
    src = f"{INCOMING_PREFIX}{fname}"
    dst = f"{PROCESSED_PREFIX}{fname}"
    s3.copy_object(Bucket=SOURCE_BUCKET, CopySource={"Bucket": SOURCE_BUCKET, "Key": src}, Key=dst)
    s3.delete_object(Bucket=SOURCE_BUCKET, Key=src)
    if i % 50 == 0:
        print(f"  moved {i}/{len(to_move)}")

print("ALL FILES MOVED TO processed/")
print("Remaining in incoming:", len(list_incoming_arc_files()))   # expect 0
